# VeriLLM: Phase 7 — RAGTruth Fine-Tuning Pipeline (GPU Accelerated)

**Project**: VeriLLM — A Hybrid, Model-Agnostic Framework for Fine-Grained LLM Hallucination Detection, Quantification and Root Cause Analysis

**Goal**: Fine-tune our own lightweight cross-encoder model on the RAGTruth benchmark for 3-class sentence-level hallucination detection:
- `0: SUPPORTED_PROXY` (Document-grounded factual assertion)
- `1: CONTRADICTED` (Evident or subtle factual contradiction of reference context)
- `2: UNSUPPORTED` (Evident or subtle baseless entity/relation/predicate not grounded in context)

**Selected Architecture**: `microsoft/deberta-v3-small` or `cross-encoder/nli-deberta-v3-xsmall` (~70M parameters).
Small, fast for local CPU inference, and superior at disentangled attention for premise-hypothesis alignment.

## 1. Environment & Hardware Verification

In [ ]:
import torch
import os
import sys

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    !nvidia-smi
else:
    print("WARNING: No GPU detected. In Colab, go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")

## 2. Install Required Dependencies

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn evaluate sentencepiece

## 3. Mount Google Drive or Upload Preprocessed Datasets
The preprocessed files (`ragtruth_train.jsonl`, `ragtruth_val.jsonl`, `ragtruth_test.jsonl`) generated by VeriLLM's `dataset_preprocessor.py` can be loaded directly from Google Drive or uploaded to `/content/data/`.

In [ ]:
import json
from pathlib import Path
from google.colab import drive

# Option A: Mount Google Drive if datasets are in Drive
try:
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/VeriLLM/data/processed')
except Exception:
    DATA_DIR = Path('/content/data')
    DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Target Data Directory: {DATA_DIR}")
print("Ensure 'ragtruth_train.jsonl' and 'ragtruth_val.jsonl' exist in this directory.")

## 4. Dataset Loader with Context Windowing & Inverse-Frequency Class Weights
To prevent truncation of the candidate assertion, the sequence packing strategy:
1. Preserves the candidate sentence 100% at the end of the sequence: `[CLS] Premise Context [SEP] Candidate Sentence [SEP]`
2. Windows the premise context to 350 words, ensuring total length strictly fits within the 512-token limit.
3. Computes inverse frequency weights to counteract the severe class imbalance (~88.8% Supported vs. 3.7% Contradicted, 7.1% Unsupported).

In [ ]:
from collections import Counter
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

LABEL2ID = {
    "CONTRADICTED": 0,
    "SUPPORTED_PROXY": 1,
    "UNSUPPORTED": 2
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

class RAGTruthFineTuningDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_length=512, max_context_words=350, max_samples=None):
        self.examples = []
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_context_words = max_context_words
        
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                item = json.loads(line)
                lbl = item.get('primary_label')
                if lbl in LABEL2ID:
                    self.examples.append(item)
                if max_samples and len(self.examples) >= max_samples:
                    break
                    
    def __len__(self):
        return len(self.examples)
        
    def __getitem__(self, idx):
        item = self.examples[idx]
        ctx = item['context_text']
        sent = item['sentence_text']
        label = LABEL2ID[item['primary_label']]
        
        # Window context if overly long, preserving complete sentence
        words = ctx.split()
        if len(words) > self.max_context_words:
            ctx = ' '.join(words[:self.max_context_words])
            
        encoding = self.tokenizer(
            ctx,
            sent,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Calculate balanced class weights
def compute_class_weights(dataset):
    counts = Counter(ex['primary_label'] for ex in dataset.examples)
    total = len(dataset.examples)
    n_classes = len(LABEL2ID)
    weights = []
    for i in range(n_classes):
        lbl = ID2LABEL[i]
        w = total / (n_classes * max(counts[lbl], 1))
        weights.append(w)
    weights = torch.tensor(weights, dtype=torch.float32)
    weights = weights / weights.sum() * n_classes
    return weights, counts

## 5. Model Initialization & Weighted Loss Function

In [ ]:
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, AutoConfig

MODEL_NAME = "cross-encoder/nli-deberta-v3-xsmall"
# Alternative: "microsoft/deberta-v3-small" (44M params) or "microsoft/deberta-v3-base" (86M params)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model loaded onto {device}. Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Training Pipeline with Macro-F1 Checkpointing

In [ ]:
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score
from torch.cuda.amp import autocast, GradScaler
import time

# Hyperparameters
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
EPOCHS = 3
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

train_path = DATA_DIR / 'ragtruth_train.jsonl'
val_path = DATA_DIR / 'ragtruth_val.jsonl'

if not train_path.exists():
    print(f"ERROR: {train_path} not found. Please upload dataset files first.")
else:
    print("Loading train and validation datasets...")
    train_ds = RAGTruthFineTuningDataset(train_path, tokenizer, max_context_words=350)
    val_ds = RAGTruthFineTuningDataset(val_path, tokenizer, max_context_words=350)
    
    class_weights, counts = compute_class_weights(train_ds)
    print(f"Train counts : {counts}")
    print(f"Loss weights : {class_weights.tolist()}")
    
    loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(device))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    scaler = GradScaler()
    
    best_val_macro_f1 = 0.0
    save_dir = Path("/content/best_verillm_deberta")
    save_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Starting training for {EPOCHS} epochs ({total_steps} total optimizer steps)...")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0
        t0 = time.time()
        
        for step, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            with autocast():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = loss_fn(outputs.logits, labels)
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            train_loss += loss.item()
            if (step + 1) % 500 == 0:
                print(f"Epoch {epoch}/{EPOCHS} | Step {step+1}/{len(train_loader)} | Running Loss: {train_loss/(step+1):.4f}")
                
        avg_train_loss = train_loss / len(train_loader)
        epoch_time = time.time() - t0
        
        # Validation Phase
        model.eval()
        all_preds, all_labels = [], []
        val_loss = 0.0
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = loss_fn(outputs.logits, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
                
        avg_val_loss = val_loss / len(val_loader)
        macro_f1 = f1_score(all_labels, all_preds, average='macro')
        
        print(f"\n[Epoch {epoch} Complete] Time: {epoch_time:.1f}s | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro-F1: {macro_f1*100:.2f}%")
        
        if macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = macro_f1
            print(f"-> New best Macro-F1! Saving checkpoint to {save_dir}...")
            model.save_pretrained(save_dir)
            tokenizer.save_pretrained(save_dir)

## 7. Comprehensive Model Evaluation & Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

target_names = [ID2LABEL[i] for i in range(len(ID2LABEL))]
print("=" * 60)
print("FINAL VALIDATION SET CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(all_labels, all_preds, target_names=target_names, digits=4))

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix (Rows=True, Cols=Pred):")
print(cm)

## 8. Export Fine-Tuned Model Weights
Zip and download the fine-tuned weights for deployment into VeriLLM's local claim verification engine.

In [ ]:
!zip -r /content/verillm_deberta_ragtruth.zip /content/best_verillm_deberta
print("Fine-tuned model successfully archived to /content/verillm_deberta_ragtruth.zip")
print("You can download this zip file and place it in backend/ml_engine/models/verillm_deberta_ragtruth/")